In [1]:
%pip install ollama


   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------------------- ------- 1.6/2.0 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 7.2 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.19
    Uninstalling pydantic-1.10.19:
      Successfully uninstalled pydantic-1.10.19


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.95.2 requires pydantic!=1.7,!=1.7.1,!=1.7.2,!=1.7.3,!=1.8,!=1.8.1,<2.0.0,>=1.6.2, but you have pydantic 2.11.1 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import ollama

def query_deepseek(prompt):
    response = ollama.chat(
        model="deepseek-r1:7b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

# Test it
resume_text = query_deepseek("Generate a well-structured ATS-friendly resume for a Software Engineer.")
print(resume_text)


<think>
Alright, so the user is asking me to generate an ATS-friendly resume for a Software Engineer. Hmm, I need to figure out what exactly they're looking for. I know that ATS stands for Application Tracking Systems, and they use these tools to filter resumes based on keywords. So, the key here is to make sure the resume is optimized for those systems.

First, I should consider the structure of a typical ATS-friendly CV. It usually includes sections like Professional Summary, Key Skills, Professional Experience, Education, Certifications, etc. Each section needs to be concise and keyword-rich without being too verbose.

I wonder about the user's background. Are they applying for entry-level positions or mid-career roles? Since it's a general request, I'll aim for a balance but maybe include some sections that are more relevant to both. They might not have specified, so I'll cover areas like programming languages, technologies, projects, and soft skills.

The resume should highlight t

In [3]:
import ollama
import json
import re

# User-provided resume data
user_data = {
    "fullName": "Aryan Naik",
    "jobTitle": "Software Engineer",
    "contact": "+91 9876543210",
    "location": "Mumbai, India",
    "email": "aryan.naik@example.com",
    "experience": [
      {
        "company": "TechCorp",
        "role": "Software Engineer",
        "place": "Mumbai, India",
        "duration": "01/06/2020 - Present",
      },
      {
        "company": "StartupX",
        "role": "Intern",
        "place": "Bangalore, India",
        "duration": "01/01/2020 - 31/05/2020",
      }
    ],
    "education": [
      {
        "institution": "Ramrao Adik Institute of Technology",
        "degree": "BTech in AI & Data Science",
        "year_end": "2026",
      }
    ],
    "skills": ["Software Development", "Machine Learning", "Data Science", "Python", "TensorFlow", "React.js", "Node.js", "Leadership", "Communication", "Problem-Solving", "Teamwork", "Leadership"],
}

# Strict prompt enforcing provided details
prompt = f"""
You are an AI that generates **STRICTLY VALID JSON ONLY** for a resume.
**DO NOT MODIFY OR OMIT ANY PROVIDED DATA.**  
NO MARKDOWN. NO EXPLANATIONS. **ONLY JSON OUTPUT!**

---
🚨 **STRICT RULES** 🚨
1️⃣ **Use exact personal details as provided. DO NOT CHANGE NAME, EMAIL, PHONE, OR COMPANY.**
2️⃣ **Experience must exactly match the provided role, company, and dates.**
3️⃣ **Summary must be 4-5 sentences based on given skills and experience.**
4️⃣ **Each experience must have at least 3 bullet points describing work done.**
5️⃣ **Education must include a one-sentence description.**
6️⃣ **All skills must be categorized under:**
   - `"Industrial Knowledge"`  
   - `"Tools & Technologies"`  
   - `"Soft Skills"`

---
✅ **EXPECTED JSON FORMAT** (STRICTLY FOLLOW THIS)
{{
    "fullName": "{user_data['fullName']}",
    "jobTitle": "{user_data['jobTitle']}",
    "contact": "{user_data['contact']}",
    "location": "{user_data['location']}",
    "email": "{user_data['email']}",
    "summary": "FILL_THIS_IN (4-5 sentences about experience, skills, and achievements).",
    "experience": [
        {{
            "company": "{user_data['experience'][0]['company']}",
            "role": "{user_data['experience'][0]['role']}",
            "place": "{user_data['experience'][0]['place']}",
            "duration": "{user_data['experience'][0]['duration']}",
            "description": [
                "FILL_THIS_IN (Key achievement or responsibility).",
                "FILL_THIS_IN (Another key contribution).",
                "FILL_THIS_IN (One more impactful point)."
            ]
        }}
    ],
    "education": [
        {{
            "institution": "{user_data['education'][0]['institution']}",
            "degree": "{user_data['education'][0]['degree']}",
            "year_end": "{user_data['education'][0]['year_end']}",
            "description": "FILL_THIS_IN (One sentence about coursework or achievements)."
        }}
    ],
    "skills": [
        {{
            "category": "Industrial Knowledge",
            "skills": ["FILL_THIS_IN (Technical skills related to the field)"]
        }},
        {{
            "category": "Tools & Technologies",
            "skills": ["FILL_THIS_IN (Technical skills related to tools and technologies)"]
        }},
        {{
            "category": "Soft Skills",
            "skills": ["FILL_THIS_IN (Soft skills like teamwork, leadership, etc.)"]
        }}
    ]
}}

---
🚨 **IMPORTANT:** 🚨
- **DO NOT RETURN ANYTHING ELSE EXCEPT PURE JSON.**
- **NO MARKDOWN (` ```json `), NO TEXT, NO EXPLANATIONS, NO COMMENTS, JUST JSON!**
- **DO NOT include ```json or any kind of markdown. If you do, the response will be REJECTED. JSON ONLY.**
- **FIRST SORT THE ALREADY PROVIDED SKILLS INTO THE CATEGORIES AND THEN YOU MIGHT ADD NEW IF REQUIRED**
- **Let the WORD COUNT be between 300 and 1000**
- **Avoid Personal Pronouns.**
- **Keep the VOCABULARY LEVEL ABOUVE AVERAGE.**
- **Keep the READABILITY LEVEL AVERAGE.**
- **USE FEW INDUSTRY-RELEVANT JARGONS.**
"""

# Query DeepSeek-R1
response = ollama.chat(model="deepseek-r1:7b", messages=[{"role": "user", "content": prompt}])

# Extract text safely
output_text = response.get("message", {}).get("content", "").strip()

if not output_text:
    print("❌ Error: No 'message' or 'content' found in response.")
else:
    # Remove unwanted AI-generated sections
    output_text = re.sub(r"<think>.*?</think>", "", output_text, flags=re.DOTALL)
    output_text = re.sub(r"^```json\s*|\s*```$", "", output_text.strip(), flags=re.MULTILINE).strip()

    # Validate JSON output
    try:
        resume_json = json.loads(output_text)
        print("✅ Successfully Parsed JSON:", json.dumps(resume_json, indent=4))
    except json.JSONDecodeError:
        print("❌ Error: DeepSeek returned invalid JSON.")
        print("Raw Output:", output_text)


✅ Successfully Parsed JSON: {
    "fullName": "Aryan Naik",
    "jobTitle": "Software Engineer",
    "contact": "+91 9876543210",
    "location": "Mumbai, India",
    "email": "aryan.naik@example.com",
    "summary": "With a strong foundation in AI & Data Science, I have excelled as a Software Engineer at TechCorp. My expertise lies in developing scalable solutions using Python and TensorFlow. I've successfully implemented machine learning models to optimize company processes.",
    "experience": [
        {
            "company": "TechCorp",
            "role": "Software Engineer",
            "place": "Mumbai, India",
            "duration": "01/06/2020 - Present",
            "description": [
                "Developed and deployed machine learning models to improve operational efficiency.",
                "Collaborated with cross-functional teams to design scalable solutions.",
                "Maintained and updated existing software systems for optimal performance."
            